In [1]:
import os
import multiprocessing
import pandas as pd
from pathlib import Path
from joblib import Parallel, delayed

#   경로 내 파일 확인
path_001 = r'./data'

#   경로 내 해당 양식 파일명 탐색
df_list_001 = list(Path(path_001).rglob('df_tokenized*.pkl'))

#   1. 경로 내 pickle파일 일괄 불러오기 함수 생성
def read_pkl_01(file_01):
    return pd.read_pickle(file_01)

#   2. 병렬 처리 설정
cl = multiprocessing.cpu_count() - 1

#   해당 경로 내 파일을 list 형태로 불러오기
df_ls_001 = Parallel(n_jobs = cl)(delayed(read_pkl_01)(file_01) for file_01 in df_list_001)

In [2]:
#   list 형태의 파일 내용을 행 기반 데이터 프레임 형태로 결합
df_001 = pd.concat(df_ls_001, axis = 0, ignore_index=True)

df_001.drop(['Compare_Date'], axis = 1, inplace = True)
df_001.info(), df_001['Target_Date'].describe()

<class 'pandas.DataFrame'>
RangeIndex: 659878 entries, 0 to 659877
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   섹션           659878 non-null  int64         
 1   제목           659878 non-null  object        
 2   언론사          659878 non-null  object        
 3   본문           659878 non-null  object        
 4   Target_Date  659878 non-null  datetime64[ns]
 5   수정           659878 non-null  object        
 6   tokens       659878 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(5)
memory usage: 35.2+ MB


(None,
 count                           659878
 mean     2026-03-18 23:20:03.527925248
 min                2026-01-02 00:00:00
 25%                2026-02-09 00:00:00
 50%                2026-03-19 00:00:00
 75%                2026-04-24 00:00:00
 max                2026-06-04 00:00:00
 Name: Target_Date, dtype: object)

In [3]:
#   필요날짜 이후 시장거래일 데이터 추출
#   기준 파일 목록 추출
check_file_001 = list(Path(path_001).rglob('KOSPI_raw_data_*.csv'))
#   최근파일 추출
check_file_002 = check_file_001[-1]
print(check_file_002)

#   해당 파일 불러오기
check_date_001 = pd.read_csv(check_file_002)
check_date_001

data\KOSPI_raw_data_251222_260605.csv


,Price,Close,High,Low,Open,Volume
0,Ticker,^KS11,^KS11,^KS11,^KS11,^KS11
1,Date,NaN,NaN,NaN,NaN,NaN
2,2025-12-22,4105.93017578125,4105.93017578125,4083.1298828125,4096.259765625,315600
3,2025-12-23,4117.31982421875,4140.83984375,4110.25,4127.39990234375,385800
4,2025-12-24,4108.6201171875,4137.2001953125,4106.6201171875,4136.240234375,363600
...,...,...,...,...,...,...
105,2026-05-28,8185.2900390625,8253.599609375,7841.009765625,8165.72998046875,669000
106,2026-05-29,8476.150390625,8476.150390625,8273.740234375,8384.3095703125,735100
107,2026-06-01,8788.3798828125,8874.16015625,8485.669921875,8485.669921875,636200
108,2026-06-02,8801.490234375,8933.6201171875,8503.1201171875,8883.1904296875,632600


In [4]:
#   열이름 변경
check_date_001_col = check_date_001.columns.tolist()
check_date_001_col[0] = 'Date'
check_date_001.columns = check_date_001_col
check_date_001.columns

#   1~2행 제거
check_date_001 = check_date_001.drop([0, 1], axis=0)
check_date_001.head(), check_date_001.tail()

(         Date             Close              High               Low  \
 2  2025-12-22  4105.93017578125  4105.93017578125   4083.1298828125   
 3  2025-12-23  4117.31982421875     4140.83984375           4110.25   
 4  2025-12-24   4108.6201171875   4137.2001953125   4106.6201171875   
 5  2025-12-26  4129.68017578125  4143.14013671875  4116.52978515625   
 6  2025-12-29  4220.56005859375  4220.56005859375  4146.47998046875   
 
                Open  Volume  
 2    4096.259765625  315600  
 3  4127.39990234375  385800  
 4    4136.240234375  363600  
 5   4130.3701171875  510200  
 6  4146.47998046875  502400  ,
            Date            Close             High              Low  \
 105  2026-05-28  8185.2900390625   8253.599609375   7841.009765625   
 106  2026-05-29   8476.150390625   8476.150390625   8273.740234375   
 107  2026-06-01  8788.3798828125    8874.16015625   8485.669921875   
 108  2026-06-02   8801.490234375  8933.6201171875  8503.1201171875   
 109  2026-06-04    8639

In [5]:
df_001['Target_Date'] = pd.to_datetime(df_001['Target_Date']).astype('datetime64[ns]')
check_date_001['Date'] = pd.to_datetime(check_date_001['Date']).astype('datetime64[ns]')

#   기준 파일을 통한 개장일 날짜 추출
trading_date = pd.DataFrame({'Compare_Date': check_date_001['Date'].unique()})
trading_date = trading_date.sort_values('Compare_Date')
df_001 = df_001.sort_values('Target_Date')

df_news = pd.merge_asof(
    df_001, trading_date, left_on = 'Target_Date', right_on = 'Compare_Date', direction = 'forward'
)

df_news.drop(['Compare_Date'], axis = 1, inplace = True)
df_news.tail()

,섹션,제목,언론사,본문,Target_Date,수정,tokens
659873,261,경북에 헴프·저속車·전기선박 규제자유특구 생긴다,매일신문,"중기부, 바이오·모빌리티·기후테크 등 전국 7개 특구 신규 지정 추진 경북에 대마 ...",2026-06-04,중기부 바이오 모빌리티 기후테크 등 전국 7개 특구 신규 지정 추진 경북에 대마 ...,"[중기부, 바이오, 모빌리티, 기후, 테크, 전국, 특구, 신규, 지정, 추진, 경..."
659874,261,"“가격 비싸, 韓제품 원해…글로벌몰 돌려달라” 美올리브영 현지몰 론칭에 ‘시끌’",헤럴드경제,[CJ올리브영 제공] [헤럴드경제=문영규 기자] CJ올리브영이 미국 1호 매장 문을...,2026-06-04,CJ올리브영 제공 헤럴드경제 문영규 기자 CJ올리브영이 미국 1호 매장 문을 ...,"[올리브, 영, 제공, 헤럴드, 경제, 문영규, 기자, 올리브, 영, 미국, 매장,..."
659875,261,"SKT, 앤트로픽 '프로젝트 글래스윙' 합류…'미토스' 접근 권한 확보",한국경제TV,SK텔레콤이 앤트로픽의 AI 보안 협력체 '프로젝트 글래스윙'에 합류한다고 4일 밝...,2026-06-04,SK텔레콤이 앤트로픽의 AI 보안 협력체 프로젝트 글래스윙 에 합류한다고 4일 밝...,"[SK텔레콤, 앤트로픽, 보안, 협력체, 프로젝트, 글래스, 윙, 합류, 밝히, 이..."
659876,261,생산·충전·모빌리티 연계…수소 생태계 승부수 띄웠다,서울경제,"■加잠수함 수주 비밀병기 ‘수소 트럭’ 완성차 공장 요구에 新동력 제안 현대차, 특...",2026-06-04,잠수함 수주 비밀병기 수소 트럭 완성차 공장 요구에 동력 제안 현대차 특사단...,"[잠수함, 수주, 비밀, 병기, 수소, 트럭, 완성차, 공장, 요구, 동력, 제안,..."
659877,771,"""미래전쟁 뼈대 구축할 것""…'드론판 킬체인' 만드는 K-스타트업",머니투데이,[스타트UP스토리] 이도경 본에이아이 대표이사 [이 기사에 나온 스타트업에 대한 보...,2026-06-04,스타트UP스토리 이도경 본에이아이 대표이사 이 기사에 나온 스타트업에 대한 보다...,"[스타트, 스토리, 이도경, 본에이아이, 대표, 이사, 기사, 나오, 스타트업, 대..."


In [9]:
#   기준 파일을 통한 개장일 날짜 추출
trading_date = pd.DataFrame({'Compare_Date': check_date_001['Date'].unique()})
trading_date = trading_date.sort_values('Compare_Date')
df_001 = df_001.sort_values('Target_Date')

df_news = pd.merge_asof(
    df_001, trading_date, left_on = 'Target_Date', right_on = 'Compare_Date', direction = 'forward'
)

df_news.drop(['Compare_Date'], axis = 1, inplace = True)
df_news.tail()

,섹션,제목,언론사,본문,Target_Date,수정,tokens
659873,771,美 뉴욕서 K-테크 스타트업 글로벌 투자 교두보 마련,아시아경제,"중진공·한국벤처투자·창업진흥원·한국관광공사, 데모데이 공동 개최 미국 뉴욕의 특화 ...",2026-06-04,중진공 한국벤처투자 창업진흥원 한국관광공사 데모데이 공동 개최 미국 뉴욕의 특화 ...,"[중진공, 한국벤처투자, 창업진흥원, 한국관광공사, 데모, 데이, 공동, 개최, 미..."
659874,771,해외여행 찬물 뿌린 유류할증료…코로나 이후 항공운송 최대폭 감소,문화일보,인천공항 전망대에서 바라본 인천국제공항 활주로. 연합뉴스 중동전쟁이 촉발한 유가 상...,2026-06-04,인천공항 전망대에서 바라본 인천국제공항 활주로 연합뉴스 중동전쟁이 촉발한 유가 상...,"[인천, 공항, 전망대, 바라보, 인천국제공항, 활주로, 연합뉴스, 중동, 전쟁, ..."
659875,263,"노동부, 한화에어로 서울본사·대전사업장 압수수색",SBS Biz,[23일 오전 대전지방고용노동청 관계자들이 대전 대덕구 안전공업 본사를 압수수색 하...,2026-06-04,23일 오전 대전지방고용노동청 관계자들이 대전 대덕구 안전공업 본사를 압수수색 하기...,"[오전, 대전, 지방, 고용, 노동청, 관계자, 대전, 대덕구, 안전공업, 본사, ..."
659876,263,"구윤철 부총리, 시장상황점검회의 주재",연합뉴스,(서울=연합뉴스) 구윤철 부총리 겸 재정경제부 장관이 4일 서울 종로구 정부서울청사...,2026-06-04,서울 연합뉴스 구윤철 부총리 겸 재정경제부 장관이 4일 서울 종로구 정부서울청사에...,"[서울, 연합뉴스, 구윤철, 부총리, 재정, 경제, 부, 장관, 서울, 종로구, 정..."
659877,771,"""미래전쟁 뼈대 구축할 것""…'드론판 킬체인' 만드는 K-스타트업",머니투데이,[스타트UP스토리] 이도경 본에이아이 대표이사 [이 기사에 나온 스타트업에 대한 보...,2026-06-04,스타트UP스토리 이도경 본에이아이 대표이사 이 기사에 나온 스타트업에 대한 보다...,"[스타트, 스토리, 이도경, 본에이아이, 대표, 이사, 기사, 나오, 스타트업, 대..."


In [10]:
#   지정 일자별 파일 분리 함수 생성
def divide_by_date(df_01, start_date, end_date, col_01 = 'Target_Date'):
    #   날짜 범위별 파일 생성
    df_02 = df_01[(df_01[col_01] >= start_date) & (df_01[col_01] < end_date)]

    #   범위별 파일명 지정
    df_02[col_01] = pd.to_datetime(df_02[col_01])

    min_date = df_02[col_01].min().strftime('%y%m%d')
    max_date = df_02[col_01].max().strftime('%y%m%d')

    #   경로 지정 및 파일 저장
    file_01 = os.path.join(path_001,  f'df_tokenized_{min_date}_{max_date}.pkl')
    df_02.to_pickle(file_01)

    return df_02


In [11]:
#   파일 분리 및 파일 저장
df_until_MAR  = divide_by_date(df_news, '2026-01-01', '2026-04-01')
df_until_JUNE  = divide_by_date(df_news, '2026-04-01', '2026-07-01')

In [12]:
#   파일 생성 확인
list(Path(os.path.join(path_001)).rglob('df_tokenized_*.pkl'))

[WindowsPath('data/df_tokenized_260102_260331.pkl'),
 WindowsPath('data/df_tokenized_260401_260604.pkl')]